# Licenta agri

Citire foi din excel

In [10]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

foi = pd.read_excel("dataset/Date_UTCN(1).xlsx", sheet_name=None)

print(list(foi.keys()))

['Air_Temperature', 'Relative_Humidity', 'Air_Pressure', 'Vapor_Pressure', 'Precipitation', 'Radiation', 'Temperature_1', 'Humidity_1', 'Wind_Speed', 'Wind direction']


Curatare coloane de spatii + unire

In [18]:
cols = ["device", "measure", "units", "value", "probe_serial", "time"]

lista = []
for nume, df in foi.items():
    df = df.copy()
    df.columns = df.columns.str.strip()
    lista.append(df[cols].assign(foaie=nume))

data = pd.concat(lista, ignore_index=True)
data.head()

,device,measure,units,value,probe_serial,time,foaie
0,BeiaTest2,temperature,celsius,255,ONS001B002A3234383114473231,2026-07-14 09:41:13,Air_Temperature
1,BeiaTest2,temperature,celsius,264,ONS001B002A3234383114473231,2026-07-14 09:26:13,Air_Temperature
2,BeiaTest2,temperature,celsius,246,ONS001B002A3234383114473231,2026-07-14 09:11:13,Air_Temperature
3,BeiaTest2,temperature,celsius,233,ONS001B002A3234383114473231,2026-07-14 08:56:11,Air_Temperature
4,BeiaTest2,temperature,celsius,232,ONS001B002A3234383114473231,2026-07-14 08:41:14,Air_Temperature


Curatarea valorilor de spatii

In [19]:
for c in ["device", "measure", "units", "probe_serial"]:
    data[c] = data[c].astype(str).str.strip()

### Rename la valorile din coloana "device" a fiecarei foi cu un id

In [36]:
data["device_id"] = data["device"].astype("category").cat.codes + 1
print(data["device_id"].isna().sum())   # 0

0


In [49]:
devices = data[["device_id", "device"]].drop_duplicates().set_index("device_id")["device"]

print(devices[1])
print(devices[2])

BeiaTest1
BeiaTest2


### Verificare coloana timp


In [21]:
data["time"] = pd.to_datetime(data["time"].astype(str).str.strip(), format="ISO8601")

In [27]:
print(data["time"].dtype)

datetime64[us]


In [28]:
print(data["time"].min(), data["time"].max())

2025-07-14 12:50:31 2026-07-14 09:41:13


In [29]:
print(data["time"].isna().sum())

0


In [30]:
print(data["time"].dt.month.value_counts().sort_index())
print(data["time"].dt.day.value_counts().sort_index())

time
1      1490
2      1344
3     23368
4     24488
5     24892
6     23206
7     61304
8     75266
9     69176
10    69400
11    45722
12     1680
Name: count, dtype: int64
time
1     13360
2     13250
3     13590
4     14366
5     14544
6     14550
7     14282
8     13968
9     13976
10    13968
11    13974
12    11800
13    11864
14    12526
15    14534
16    13736
17    13468
18    13776
19    14096
20    14120
21    14112
22    14140
23    14082
24    14072
25    14072
26    14064
27    13922
28    13356
29    13296
30    13296
31     9176
Name: count, dtype: int64


Distanta dintre fiecare masuratoare

In [44]:
t = data[(data["device_id"] == 2) & (data["foaie"] == "Air_Temperature")]["time"].sort_values()
print(len(t))
print(t.diff().value_counts().head())

48514
time
0 days 00:05:00    23489
0 days 00:15:00     9434
0 days 00:04:59     4481
0 days 00:05:01     1667
0 days 00:04:57     1255
Name: count, dtype: int64


### Rename la valorile din coloana "measure" a fiecarei foi cu un id

In [50]:
print(data.groupby("foaie")["measure"].unique())

foaie
Air_Pressure                                  [pressure]
Air_Temperature                            [temperature]
Humidity_1           [moisture_without_salinity_depth_5]
Precipitation                            [precipitation]
Radiation                                    [radiation]
Relative_Humidity                             [humidity]
Temperature_1                      [temperature_depth_5]
Vapor_Pressure                          [pressure_vapor]
Wind direction                          [vane_direction]
Wind_Speed                                  [wind_speed]
Name: measure, dtype: object
